# ⚡ Z-Fooocus
**Multi-Model Studio** — Z-Image Turbo · FLUX.2-klein · Qwen-Image-Edit

Generate · Img2Img · Inpaint with model switching

### 🔑 Setup: Add your HuggingFace token
1. Go to https://huggingface.co/settings/tokens → Create a token (Read access)
2. In Colab: Click 🔑 icon (left sidebar) → Add secret: Name = `HF_TOKEN`, Value = your token
3. Enable notebook access toggle
4. Restart runtime, then run Step 1

In [ ]:
# ── Step 1: Install & Download Models ─────────────────────
import os

# ── HF Token ──────────────────────────────────────────────
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('🔑 HF token loaded from Colab secrets')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', None)
    if HF_TOKEN:
        print('🔑 HF token loaded from environment')
    else:
        print('⚠️ No HF token found. Some downloads may fail.')
        print('   Add HF_TOKEN to Colab secrets (🔑 icon in sidebar)')

# ── ComfyUI ───────────────────────────────────────────────
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    %cd /content/ComfyUI
    !pip install -q -r requirements.txt

# ComfyUI-GGUF custom nodes
if not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI-GGUF'):
    !git clone https://github.com/city96/ComfyUI-GGUF.git /content/ComfyUI/custom_nodes/ComfyUI-GGUF
    !pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt

!pip install -q rembg[gpu] onnxruntime-gpu huggingface_hub

# ── Download helper ───────────────────────────────────────
UNET = '/content/ComfyUI/models/unet'
CLIP = '/content/ComfyUI/models/clip'
VAE  = '/content/ComfyUI/models/vae'
os.makedirs(UNET, exist_ok=True)
os.makedirs(CLIP, exist_ok=True)
os.makedirs(VAE,  exist_ok=True)

from huggingface_hub import hf_hub_download

def dl(repo, filename, dest_dir, dest_name=None):
    name = dest_name or os.path.basename(filename)
    dest = os.path.join(dest_dir, name)
    if os.path.exists(dest):
        print(f'  ✅ {name}')
        return
    print(f'  ⏳ {name}...')
    path = hf_hub_download(
        repo_id=repo, filename=filename,
        local_dir='/tmp/hf_dl', token=HF_TOKEN
    )
    os.makedirs(dest_dir, exist_ok=True)
    os.rename(path, dest)
    print(f'  ✅ {name}')

# ── Z-Image Turbo ─────────────────────────────────────────
print('\n📦 Z-Image Turbo FP8')
dl('T5B/Z-Image-Turbo-FP8', 'z-image-turbo-fp8-e4m3fn.safetensors', UNET)
dl('Comfy-Org/Qwen_3_4B_Lumina2_CLIP', 'qwen_3_4b.safetensors', CLIP)
dl('Comfy-Org/flux1-dev', 'ae.safetensors', VAE)

# ── FLUX.2-klein FP8 ──────────────────────────────────────
print('\n📦 FLUX.2-klein 9B FP8 (~10GB)')
dl('black-forest-labs/FLUX.2-klein-9b-kv-fp8', 'flux-2-klein-9b-kv-fp8.safetensors', UNET)

# ── Qwen-Image-Edit-2511 GGUF ─────────────────────────────
print('\n📦 Qwen-Image-Edit-2511 GGUF Q4_K_M')
dl('unsloth/Qwen-Image-Edit-2511-GGUF', 'qwen-image-edit-2511-Q4_K_M.gguf', UNET)
dl('unsloth/Qwen2.5-VL-7B-Instruct-GGUF', 'Qwen2.5-VL-7B-Instruct-UD-Q4_K_XL.gguf', CLIP)
dl('Qwen/Qwen-Image-Edit-2511', 'vae/diffusion_pytorch_model.safetensors', VAE, 'qwen_image_vae.safetensors')

print('\n🎉 All models ready!')

In [ ]:
# ── Step 2: Launch Z-Fooocus ──────────────────────────────
import os
os.chdir('/content')
REPO = 'https://github.com/MuntasirMalek/Z-Fooocus.git'
if os.path.exists('/content/Z-Fooocus'):
    %cd /content/Z-Fooocus
    !git pull --ff-only || true
else:
    !git clone {REPO}
    %cd /content/Z-Fooocus
!python app.py